# Lesson 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是一個用於構建 AI 代理的統一框架。它提供了一個乾淨、可組合的架構，包含四個核心組件：

- **Client** – 連接到 AI 模型端點並處理通訊
- **Agent** – 將指令和工具定義包裝到客戶端中
- **Tools** – 通過模型可調用的自訂功能擴展代理能力
- **Session** – 維護多輪對話的對話歷史

在本課中，我們將使用這些概念構建一個**旅遊預訂代理**，用於檢查目的地的可用性。


## 設置


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 了解 Agent Framework 架構

Microsoft Agent Framework 採用分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – 一個 `AzureAIProjectAgentProvider` 連接到 Azure OpenAI 部署。它負責認證、請求格式化和回應解析。
2. **Agent** – 由 client 透過 `provider.create_agent()` 創建，agent 將模型存取與指示（系統提示）及工具結合。
3. **Tools** – 以 `@tool` 裝飾的 Python 函數，agent 可呼叫這些函數執行操作或檢索資料。
4. **Session** – 一個 `AgentSession` 物件（由 `agent.create_session()` 創建），用來存儲對話歷史，使 agent 能記住先前上下文，實現多輪對話。

讓我們一步一步建立每一層。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 裝飾器新增工具

工具讓代理人能執行生成文本以外的動作。`@tool` 裝飾器將普通的 Python 函數轉換成代理人能調用的東西。

重點：
- 使用 `Annotated[type, "description"]` 讓模型理解每個參數。
- 函數說明字串會成為模型看到的工具描述。
- `approval_mode="never_require"` 表示工具自動執行，無需用戶確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 建立一個具工具的代理人

現在我們將客戶端、指令和工具結合成一個代理人。`instructions` 作為系統提示 — 它們定義了代理人的角色和行為。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 多輪對話與會話

`AgentSession`（通過 `agent.create_session()` 創建）會追蹤對話中的所有訊息。通過在每次調用 `agent.run()` 時傳入相同的會話，智能代理可以訪問完整的對話歷史，並可回顧先前的訊息。

我們傳入 `tools=[check_destination_availability]`，讓智能代理在每輪對話中都可以調用我們的可用性檢查工具。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

在本課程中，你探索了 Microsoft Agent Framework 的四大支柱：

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` 使用基於憑證的身份驗證連接到 Azure OpenAI |
| **Agent** | `provider.create_agent()` 將模型連接、指令和名稱打包 |
| **Tools** | `@tool` 裝飾器公開 Python 函數供代理調用 |
| **Session** | `agent.create_session()` 維持多輪對話歷史 |

這些構建塊結合起來可創建能進行自然對話、調用外部函數並保持上下文的代理 — 為後續課程中更高級的代理模式奠定基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件係使用人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所翻譯。雖然我們致力於確保準確性，但請注意自動翻譯可能含有錯誤或不準確之處。原始文件之母語版本應被視為權威來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用本翻譯而引起之任何誤解或誤譯負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
